In [ ]:
import os
import json
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
"""Visualization and dataset overview"""


DATASET_DIR = Path("/kaggle/input/datasets/khlaifiabilel/pastis/PASTIS")

CROP_CLASSES = {
    0: "Background (Non-Agricultural)",
    1: "Meadow",
    2: "Soft Winter Wheat",
    3: "Corn",
    4: "Winter Barley",
    5: "Winter Rapeseed",
    6: "Spring Barley",
    7: "Sunflower",
    8: "Grapevine",
    9: "Beet",
    10: "Winter Triticale",
    11: "Winter Durum Wheat",
    12: "Fruits, Vegetables, Flowers",
    13: "Potatoes",
    14: "Leguminous Fodder",
    15: "Soybeans",
    16: "Orchard",
    17: "Mixed Cereal",
    18: "Sorghum",
    19: "Void Label (Outside Patch)"
}

def print_tree(startpath, max_depth=3):
    print("=" * 60)
    print("Dataset tree")
    print("=" * 60)
    for root, dirs, files in os.walk(startpath):
        level = root.replace(str(startpath), '').count(os.sep)
        if level > max_depth:
            continue
        indent = ' ' * 4 * level
        print(f'{indent}|-- {os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for i, f in enumerate(sorted(files)):
            if i < 5:
                print(f'{subindent}|-- {f}')
            elif i == 5:
                print(f'{subindent}|-- ... ({len(files)} total files)')
                break
    print("\n")

def examine_metadata(dataset_path):
    print("=" * 60)
    print("Dataset info")
    print("=" * 60)
    
    meta_path = dataset_path / "metadata.geojson"
    norm_path = dataset_path / "NORM_S2_patch.json"
    
    if meta_path.exists():
        with open(meta_path, 'r') as f:
            meta_data = json.load(f)
            features = meta_data.get('features', [])
            total_images = len(features)
            print(f"Total Unique Images in Dataset : {total_images}")
            
            if total_images > 0:
                sample_feat = features[0]['properties']
                print(f"Observation Dates per Image    : {len(sample_feat.get('dates-S2', {}))} timesteps")
                print(f"Sample Image ID                : {sample_feat.get('ID_PATCH')}")
                print(f"Fields (Участки) in Sample     : {sample_feat.get('N_Parcel')}")
    
    print("\n" + "-" * 40)
    print("Broken down groups with their correponding Mean and Standart deviation values")
    print("-" * 40)
    if norm_path.exists():
        with open(norm_path, 'r') as f:
            norm_data = json.load(f)
            for fold_name, stats in sorted(norm_data.items()):
                mean_rgb = np.round(stats['mean'][:3], 2)
                std_rgb = np.round(stats['std'][:3], 2)
                print(f"-> {fold_name} | Mean (RGB bands): {mean_rgb} | Std: {std_rgb}")
    print("=" * 60 + "\n")

def pastis_visualizer(patch_id, time_idx=15):
    data_path = list(DATASET_DIR.glob(f"DATA_S2/S2_{patch_id}.npy"))[0]
    parcel_path = list(DATASET_DIR.glob(f"ANNOTATIONS/ParcelIDs_{patch_id}.npy"))[0]
    semantic_path = list(DATASET_DIR.glob(f"ANNOTATIONS/TARGET_{patch_id}.npy"))[0]
    
    data = np.load(data_path)
    parcel_mask = np.load(parcel_path)
    crop_mask = np.load(semantic_path)[0]
    
    unique_crops = np.unique(crop_mask)
    crop_names = [CROP_CLASSES.get(int(c), f"Unknown ({c})") for c in unique_crops if c != 0]
    
    print("=" * 60)
    print(f"IMAGE DETAIL VIEW | IMAGE ID: {patch_id} | TIMESTEP: {time_idx}")
    print("=" * 60)
    print(f"Satellite Tensor Shape : {data.shape} -> (Dates, Bands, Height, Width)")
    print(f"Field Boundaries Mask  : {parcel_mask.shape} -> (Contains unique tracking integers)")
    print(f"Crop Classification    : {crop_mask.shape} -> (Contains target categories 0-19)")
    print(f"Crops Identified       : {crop_names}")
    print("-" * 60)
    
    rgb = data[time_idx, :3, :, :].transpose(1, 2, 0) 
    nir = data[time_idx, 6, :, :]                     
    red_edge = data[time_idx, 3, :, :]                
    
    def normalize_img(img):
        p2, p98 = np.percentile(img, (2, 98))
        return np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)

    fig, axes = plt.subplots(1, 5, figsize=(24, 5))
    fig.suptitle(f"Multi-spectral & Ground Truth Analysis | Image ID: {patch_id}", fontsize=16, fontweight='bold')

    axes[0].imshow(normalize_img(rgb))
    axes[0].set_title("1. RGB (Visible Light)")
    axes[0].axis('off')

    axes[1].imshow(normalize_img(nir), cmap='Greens')
    axes[1].set_title("2. NIR (Vegetation Density)")
    axes[1].axis('off')

    axes[2].imshow(normalize_img(red_edge), cmap='magma')
    axes[2].set_title("3. Red Edge (Chlorophyll)")
    axes[2].axis('off')

    axes[3].imshow(parcel_mask.squeeze(), cmap='prism', interpolation='nearest')
    axes[3].set_title("4. Field Boundaries (Участки)")
    axes[3].axis('off')
    
    axes[4].imshow(crop_mask, cmap='tab20', interpolation='nearest', vmin=0, vmax=20)
    axes[4].set_title("5. Target Map (Crop Types)")
    axes[4].axis('off')
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    print_tree(DATASET_DIR.parent, max_depth=3)
    examine_metadata(DATASET_DIR)
    
    s2_files = sorted(list(DATASET_DIR.glob("DATA_S2/S2_*.npy")))
    
    if len(s2_files) >= 2:
        patch_1 = s2_files[0].stem.split('_')[1]
        patch_2 = s2_files[1].stem.split('_')[1]
        
        pastis_visualizer(patch_1, time_idx=15)
        pastis_visualizer(patch_2, time_idx=20)

In [ ]:
patch_id = "10000"
time_idx = 15

s2_file = list(DATASET_DIR.glob(f"DATA_S2/S2_{patch_id}.npy"))[0]
target_file = list(DATASET_DIR.glob(f"ANNOTATIONS/TARGET_{patch_id}.npy"))[0]
parcel_file = list(DATASET_DIR.glob(f"ANNOTATIONS/ParcelIDs_{patch_id}.npy"))[0]
instances_file = list(DATASET_DIR.glob(f"INSTANCE_ANNOTATIONS/INSTANCES_{patch_id}.npy"))[0]
heatmap_file = list(DATASET_DIR.glob(f"INSTANCE_ANNOTATIONS/HEATMAP_{patch_id}.npy"))[0]
zones_file = list(DATASET_DIR.glob(f"INSTANCE_ANNOTATIONS/ZONES_{patch_id}.npy"))[0]

s2_data = np.load(s2_file)
crop_target = np.load(target_file)[0]
parcel_registry = np.load(parcel_file).squeeze()
local_instances = np.load(instances_file).squeeze()
heatmap_data = np.load(heatmap_file).squeeze()
zones_data = np.load(zones_file).squeeze()

rgb = s2_data[time_idx, :3, :, :].transpose(1, 2, 0)
p2, p98 = np.percentile(rgb, (2, 98))
rgb_normalized = np.clip((rgb - p2) / (p98 - p2 + 1e-8), 0, 1)

fig, axes = plt.subplots(1, 6, figsize=(26, 5))
fig.suptitle(f"PASTIS Complete Data & Annotation Modalities | Patch: {patch_id}", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_normalized)
axes[0].set_title("1. Raw Satellite (RGB)")
axes[0].axis("off")

axes[1].imshow(crop_target, cmap="tab20", interpolation="nearest", vmin=0, vmax=20)
axes[1].set_title("2. TARGET (Crop Types)")
axes[1].axis("off")

axes[2].imshow(parcel_registry, cmap="prism", interpolation="nearest")
axes[2].set_title("3. ParcelIDs (Gov Registry)")
axes[2].axis("off")

axes[3].imshow(local_instances, cmap="jet", interpolation="nearest")
axes[3].set_title("4. INSTANCES (Local Counters)")
axes[3].axis("off")

axes[4].imshow(heatmap_data, cmap="magma", interpolation="nearest")
axes[4].set_title("5. HEATMAP (Centerness)")
axes[4].axis("off")

axes[5].imshow(zones_data, cmap="cividis", interpolation="nearest")
axes[5].set_title("6. ZONES (Borders)")
axes[5].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class PASTIS_Dataset(Dataset):
    def __init__(self, dataset_dir, target_folds, norm_path):

        self.dataset_dir = Path(dataset_dir)
        self.target_folds = [f"Fold_{f}" for f in target_folds]
        
        with open(norm_path, 'r') as f:
            norm_data = json.load(f)
            
        self.patch_ids = []
        self.fold_means = {}
        self.fold_stds = {}
        
        for fold in self.target_folds:
            if fold in norm_data:
                self.patch_ids.extend(norm_data[fold]["patches"])
                
                self.fold_means[fold] = np.array(norm_data[fold]["mean"])
                self.fold_stds[fold] = np.array(norm_data[fold]["std"])
                
        self.patch_to_fold = {}
        for fold in self.target_folds:
            for pid in norm_data[fold]["patches"]:
                self.patch_to_fold[str(pid)] = fold

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        patch_id = str(self.patch_ids[idx])
        fold = self.patch_to_fold[patch_id]
        
        s2_path = self.dataset_dir / f"DATA_S2/S2_{patch_id}.npy"
        target_path = self.dataset_dir / f"ANNOTATIONS/TARGET_{patch_id}.npy"
        
        s2_data = np.load(s2_path).astype(np.float32)
        target_data = np.load(target_path)[0].astype(np.int64)
        
        mean = self.fold_means[fold].reshape(1, 10, 1, 1)
        std = self.fold_stds[fold].reshape(1, 10, 1, 1)
        s2_normalized = (s2_data - mean) / std
        
        return torch.from_numpy(s2_normalized), torch.from_numpy(target_data)

def pad_collate_fn(batch):
    max_t = max([item[0].shape[0] for item in batch])
    
    padded_s2_list = []
    mask_list = []
    target_list = []
    
    for s2, target in batch:
        t_current, c, h, w = s2.shape
        
        padded_s2 = torch.zeros((max_t, c, h, w), dtype=torch.float32)
        padded_s2[:t_current, :, :, :] = s2
        
        mask = torch.zeros(max_t, dtype=torch.int32)
        mask[:t_current] = 1
        
        padded_s2_list.append(padded_s2)
        mask_list.append(mask)
        target_list.append(target)
        
    batch_s2 = torch.stack(padded_s2_list, dim=0)       
    batch_masks = torch.stack(mask_list, dim=0)         
    batch_targets = torch.stack(target_list, dim=0)     
    
    return batch_s2, batch_targets, batch_masks

In [ ]:
GEOJSON_PATH = DATASET_DIR / "metadata.geojson"
NUM_CLASSES = 20

train_folds = [1, 2, 3, 4]

with open(GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

train_patch_ids = []
for feature in geojson_data["features"]:
    properties = feature["properties"]
    if properties["Fold"] in train_folds:
        train_patch_ids.append(properties["ID_PATCH"])

print(f"Found {len(train_patch_ids)} training patches inside folds {train_folds}.")
print("Scanning pixels to compute exact weight configurations...")

global_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

for patch_id in train_patch_ids:
    target_path = DATASET_DIR / f"ANNOTATIONS/TARGET_{patch_id}.npy"
    if target_path.exists():
        crop_target = np.load(target_path)[0]
        
        counts = np.bincount(crop_target.flatten(), minlength=NUM_CLASSES)
        global_pixel_counts += counts

total_pixels = np.sum(global_pixel_counts)
global_pixel_counts = np.where(global_pixel_counts == 0, 1, global_pixel_counts)

calculated_weights = total_pixels / (NUM_CLASSES * global_pixel_counts)

calculated_weights = calculated_weights / np.mean(calculated_weights)

calculated_weights[0] = min(calculated_weights[0], 0.1) 
calculated_weights[19] = 0.0

print("\n--- Final Calculated Weight Tensor Array ---")
print("Copy this array directly into your PyTorch training file:\n")
print("class_weights = torch.tensor([")
print(",\n".join([f"    {round(w, 4)}" for w in calculated_weights]))
print("], dtype=torch.float32)")

In [ ]:
class_weights = torch.tensor([
    0.0252,
    0.0551,
    0.1484,
    0.1129,
    0.5001,
    0.5839,
    1.6031,
    1.0492,
    0.3594,
    1.2424,
    1.3309,
    0.965,
    1.0971,
    3.4784,
    0.6372,
    0.9177,
    0.9466,
    2.0294,
    2.8119,
    0.0
], dtype=torch.float32)

class FocalDiceLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, alpha=0.25):
        super(FocalDiceLoss, self).__init__()
        self.weight = weight 
        self.gamma = gamma   
        self.alpha = alpha
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        
        pt = torch.exp(-ce_loss) 
        
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        
        inputs_soft = F.softmax(inputs, dim=1)
        
        targets_one_hot = F.one_hot(targets, num_classes=20).permute(0, 3, 1, 2).float()
        
        dims = (0, 2, 3)
        intersection = torch.sum(inputs_soft * targets_one_hot, dims)
        cardinality = torch.sum(inputs_soft + targets_one_hot, dims)
        
        smooth = 1e-6
        dice_score = (2. * intersection + smooth) / (cardinality + smooth)
        
        dice_loss = (1 - dice_score).mean()
        
        return focal_loss + dice_loss